# Bright LEO Photometry (single)
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Measure brightness of bright LEO object.<br>
<br>
**History**<br>
coding 2026-07-05 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Measurement settings**

In [ ]:
# aperture radius for object
aperture_radius = 24  # [pixel] | int or float

# aperture radius for background
annulus_r_in = 30     # [pixel] | int or float
annulus_r_out = 36    # [pixel] | int or float

### Import and initial settings
**PATH settings**

In [ ]:
PATH_base  = "/Users/kiyoaki/VScode/satphotometry_photometry/"
PATH_input = PATH_base + "input/"
PATH_fits  = PATH_input + "HORN-R/HORN-R_00010.fits"

PATH_output = PATH_base + "output/"
PATH_magzero = PATH_output + "magzero/magzero_2026-07-02T125622_GAIN56.json"

**Standard libraries**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from scipy import ndimage

from astropy.stats import SigmaClip
from photutils.aperture import (
    CircularAperture,
    CircularAnnulus,
    ApertureStats,
    aperture_photometry,
)

### Object detection

#### Function

In [ ]:
def detect_brightest_point_source(
    image,
    detection_sigma=8.0,
    edge_sigma=5.0,
    cutout_half_size=30,
    min_pixels=5,
):
    """
    画像中で最も明るいほぼ点像の天体を検出し、
    輝度加重重心と点像半径を求める。

    Parameters
    ----------
    image : 2D ndarray
        FITS画像データ。

    detection_sigma : float
        天体候補領域を抽出する際の背景ノイズに対する閾値。

    edge_sigma : float
        Sobelエッジを抽出する際の閾値。
        小さくすると弱いエッジまで拾う。

    cutout_half_size : int
        最も明るい画素の周囲に切り出す半幅 [pixel]。

    min_pixels : int
        天体候補として必要な最小画素数。

    Returns
    -------
    result : dict
        以下を含む辞書。

        x_centroid : float
            画像全体におけるx方向重心 [pixel]。

        y_centroid : float
            画像全体におけるy方向重心 [pixel]。

        radius : float
            エッジ画素までの距離の中央値から求めた半径 [pixel]。

        radius_std : float
            エッジ半径のばらつき [pixel]。

        equivalent_radius : float
            検出領域の面積から求めた等価半径 [pixel]。

        edge_x, edge_y : ndarray
            画像全体におけるエッジ画素座標。

        source_mask : ndarray
            切り出し画像内の人工天体候補マスク。

        cutout : ndarray
            人工天体周辺の切り出し画像。

        bounds : tuple
            切り出し範囲 (x_min, x_max, y_min, y_max)。
    """

    image = np.asarray(image, dtype=float)

    if image.ndim != 2:
        raise ValueError("imageは2次元配列である必要があります。")

    # --------------------------------------------------------
    # NaN、infを背景中央値で置換
    # --------------------------------------------------------

    finite_mask = np.isfinite(image)

    if not np.any(finite_mask):
        raise ValueError("画像中に有限値がありません。")

    global_median = np.nanmedian(image[finite_mask])

    clean_image = image.copy()
    clean_image[~finite_mask] = global_median

    # --------------------------------------------------------
    # 画像全体の背景統計
    # --------------------------------------------------------

    mean, median, std = sigma_clipped_stats(
        clean_image,
        sigma=3.0,
        maxiters=5,
    )

    if not np.isfinite(std) or std <= 0:
        raise ValueError("背景ノイズの標準偏差を推定できませんでした。")

    # --------------------------------------------------------
    # 最も明るい画素を探す
    # --------------------------------------------------------

    brightest_index = np.nanargmax(clean_image)
    brightest_y, brightest_x = np.unravel_index(
        brightest_index,
        clean_image.shape,
    )

    # --------------------------------------------------------
    # 明るい画素の周囲を切り出す
    # --------------------------------------------------------

    ny, nx = clean_image.shape

    x_min = max(0, brightest_x - cutout_half_size)
    x_max = min(nx, brightest_x + cutout_half_size + 1)

    y_min = max(0, brightest_y - cutout_half_size)
    y_max = min(ny, brightest_y + cutout_half_size + 1)

    cutout = clean_image[y_min:y_max, x_min:x_max].copy()

    # --------------------------------------------------------
    # 切り出し領域の背景を再推定
    # --------------------------------------------------------

    cut_mean, cut_median, cut_std = sigma_clipped_stats(
        cutout,
        sigma=3.0,
        maxiters=5,
    )

    if not np.isfinite(cut_std) or cut_std <= 0:
        cut_median = median
        cut_std = std

    background_subtracted = cutout - cut_median

    # --------------------------------------------------------
    # 人工天体候補の領域を抽出
    # --------------------------------------------------------

    source_threshold = detection_sigma * cut_std
    threshold_mask = background_subtracted > source_threshold

    # 小さな穴を埋める
    threshold_mask = ndimage.binary_fill_holes(threshold_mask)

    # 1画素程度の隙間をつなぐ
    threshold_mask = ndimage.binary_closing(
        threshold_mask,
        structure=np.ones((3, 3), dtype=bool),
    )

    # 連結領域に分割
    labels, number_of_labels = ndimage.label(threshold_mask)

    if number_of_labels == 0:
        raise RuntimeError(
            "人工天体候補を検出できませんでした。"
            "detection_sigmaを小さくしてください。"
        )

    # --------------------------------------------------------
    # 最も明るい画素を含む連結領域を選択
    # --------------------------------------------------------

    local_brightest_x = brightest_x - x_min
    local_brightest_y = brightest_y - y_min

    target_label = labels[local_brightest_y, local_brightest_x]

    if target_label == 0:
        # 最も明るい画素が閾値領域に入っていない場合は、
        # 総フラックスが最大の領域を選択
        label_fluxes = []

        for label_number in range(1, number_of_labels + 1):
            region = labels == label_number
            flux = np.sum(
                np.clip(background_subtracted[region], 0, None)
            )
            label_fluxes.append(flux)

        target_label = np.argmax(label_fluxes) + 1

    source_mask = labels == target_label

    if np.count_nonzero(source_mask) < min_pixels:
        raise RuntimeError(
            f"検出領域が小さすぎます："
            f"{np.count_nonzero(source_mask)} pixels"
        )

    # --------------------------------------------------------
    # 輝度加重重心
    # --------------------------------------------------------

    yy, xx = np.indices(cutout.shape)

    weights = np.clip(background_subtracted, 0, None)
    weights[~source_mask] = 0.0

    total_weight = np.sum(weights)

    if total_weight <= 0:
        raise RuntimeError("重心計算に使用できる正のフラックスがありません。")

    x_centroid_local = np.sum(xx * weights) / total_weight
    y_centroid_local = np.sum(yy * weights) / total_weight

    x_centroid = x_centroid_local + x_min
    y_centroid = y_centroid_local + y_min

    # --------------------------------------------------------
    # Sobelフィルタによるエッジ検出
    # --------------------------------------------------------

    # ノイズ抑制のため軽くGaussian平滑化
    smoothed = ndimage.gaussian_filter(
        background_subtracted,
        sigma=1.0,
    )

    gradient_x = ndimage.sobel(smoothed, axis=1, mode="reflect")
    gradient_y = ndimage.sobel(smoothed, axis=0, mode="reflect")

    gradient_magnitude = np.hypot(gradient_x, gradient_y)

    # 背景領域における勾配ノイズを推定
    background_region = ~ndimage.binary_dilation(
        source_mask,
        iterations=2,
    )

    gradient_background = gradient_magnitude[background_region]

    if gradient_background.size > 0:
        _, gradient_median, gradient_std = sigma_clipped_stats(
            gradient_background,
            sigma=3.0,
            maxiters=5,
        )
    else:
        gradient_median = np.median(gradient_magnitude)
        gradient_std = np.std(gradient_magnitude)

    edge_threshold = gradient_median + edge_sigma * gradient_std

    edge_mask_raw = gradient_magnitude > edge_threshold

    # 人工天体候補の近傍だけに限定
    source_neighborhood = ndimage.binary_dilation(
        source_mask,
        iterations=2,
    )

    edge_mask = edge_mask_raw & source_neighborhood

    # source_mask自体の境界も求める
    eroded_mask = ndimage.binary_erosion(source_mask)
    mask_boundary = source_mask & ~eroded_mask

    # Sobelエッジが少なすぎる場合に備えてマスク境界を追加
    edge_mask = edge_mask | mask_boundary

    edge_y_local, edge_x_local = np.nonzero(edge_mask)

    if len(edge_x_local) == 0:
        raise RuntimeError("人工天体のエッジを検出できませんでした。")

    # --------------------------------------------------------
    # 重心からエッジまでの距離を半径として求める
    # --------------------------------------------------------

    edge_distances = np.sqrt(
        (edge_x_local - x_centroid_local) ** 2
        + (edge_y_local - y_centroid_local) ** 2
    )

    # 異常に内側または外側のエッジを除外
    distance_median = np.median(edge_distances)
    distance_mad = np.median(
        np.abs(edge_distances - distance_median)
    )

    if distance_mad > 0:
        robust_sigma = 1.4826 * distance_mad

        valid_edge = (
            np.abs(edge_distances - distance_median)
            < 3.0 * robust_sigma
        )

        filtered_distances = edge_distances[valid_edge]
        edge_x_local = edge_x_local[valid_edge]
        edge_y_local = edge_y_local[valid_edge]
    else:
        filtered_distances = edge_distances

    radius = np.median(filtered_distances)
    radius_std = np.std(filtered_distances)

    # --------------------------------------------------------
    # 面積から求める等価半径
    # --------------------------------------------------------

    source_area = np.count_nonzero(source_mask)
    equivalent_radius = np.sqrt(source_area / np.pi)

    # エッジ座標を画像全体の座標へ変換
    edge_x = edge_x_local + x_min
    edge_y = edge_y_local + y_min

    return {
        "x_centroid": float(x_centroid),
        "y_centroid": float(y_centroid),
        "radius": float(radius),
        "radius_std": float(radius_std),
        "equivalent_radius": float(equivalent_radius),
        "source_area": int(source_area),
        "peak_value": float(clean_image[brightest_y, brightest_x]),
        "background": float(cut_median),
        "background_std": float(cut_std),
        "edge_x": edge_x,
        "edge_y": edge_y,
        "source_mask": source_mask,
        "edge_mask": edge_mask,
        "cutout": cutout,
        "background_subtracted": background_subtracted,
        "bounds": (x_min, x_max, y_min, y_max),
    }

#### Read FITS image

In [ ]:
with fits.open(PATH_fits) as img_hdu:
    img_header = img_hdu[0].header.copy()
    img_data = np.asarray(img_hdu[0].data, dtype=float)

#### Detection

In [ ]:
result = detect_brightest_point_source(
    img_data,
    detection_sigma=8.0,
    edge_sigma=5.0,
    cutout_half_size=30,
)

# print(
#     f"Centroid: "
#     f"x = {result['x_centroid']:.3f} pixel, "
#     f"y = {result['y_centroid']:.3f} pixel"
# )

# print(
#     f"Edge-based radius: "
#     f"{result['radius']:.3f} ± "
#     f"{result['radius_std']:.3f} pixel"
# )

# print(
#     f"Equivalent radius: "
#     f"{result['equivalent_radius']:.3f} pixel"
# )

### Measurement

In [ ]:
# Object position
x_object = result["x_centroid"]
y_object = result["y_centroid"]

position = (x_object, y_object)

# object region
aperture = CircularAperture(
    position,
    r=aperture_radius,
)

# background region
annulus = CircularAnnulus(
    position,
    r_in=annulus_r_in,
    r_out=annulus_r_out,
)

# aperture measurement for object
phot_table = aperture_photometry(
    img_data,
    aperture,
    method="exact",
)

aperture_sum = float(phot_table["aperture_sum"][0])

# background measurement
sigma_clip = SigmaClip(
    sigma=3.0,
    maxiters=5,
)

annulus_stats = ApertureStats(
    img_data,
    annulus,
    sigma_clip=sigma_clip,
)

background_median = float(annulus_stats.median)
aperture_area = float(aperture.area)

background_sum = background_median * aperture_area

# object net flux [ADU]
object_flux = aperture_sum - background_sum

# # [pix] => [arcsec]
# pixel_scales_deg = [6.3e-05, 6.3e-05]

# # [pix^2] => [arcsec^2]
# pixel_area_arcsec2 = (
#     pixel_scales_deg[0]
#     * pixel_scales_deg[1]
#     * 3600.0**2
# )

# aperture_area_arcsec2 = aperture_area*pixel_area_arcsec2
# object_flux_arcsec2   = aperture_sum/aperture_area_arcsec2

# object magnitude
with open(PATH_magzero, "r", encoding="utf-8") as f:
    magzero_data = json.load(f)
photometric_zeropoint_elec = magzero_data["phot_params"]["photometric_zeropoint_elec"]

instrumental_mag = -2.5 * np.log10(object_flux)
apparent_mag     = instrumental_mag + 2.5 * np.log10(img_header["EXPTIME"]) - 2.5 * np.log10(img_header["EGAINSAV"]) + photometric_zeropoint_elec

### Result

#### Summary

In [ ]:
print(f"Object position")
print(f"  x = {x_object:.4f} pixel")
print(f"  y = {y_object:.4f} pixel")

print()
print(f"Aperture radius       = {aperture_radius:.1f} pixel")
print(f"Annulus inner radius  = {annulus_r_in:.1f} pixel")
print(f"Annulus outer radius  = {annulus_r_out:.1f} pixel")

print()
print(f"Aperture area         = {aperture_area:.4f} pixel^2")
print(f"Aperture sum          = {aperture_sum:.4f} ADU")
print(f"Background median     = {background_median:.4f} ADU/pixel")
print(f"Background sum        = {background_sum:.4f} ADU")

print()
print(f"Object net flux       = {object_flux:.4f} ADU")
print(f"Instrumental mag      = {instrumental_mag:.4f} mag")
print(f"Apparent mag          = {apparent_mag:.4f} mag")